# Bronze Layer, Transactions Data Ingestion
**Source:** HDFS (`hdfs://master:8020/transaction.snappy.parquet`)  
**Target:** HDFS Delta Table `/delta/bronze/Transactions`  
**Pattern:** Full Load with Ingestion Timestamp  
**Layer:** Bronze (Raw Ingestion)

---

In [1]:
import os, sys, logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp, col, current_timestamp

os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PYTHONPATH"] = "/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip"
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.7-src.zip")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "source_path" : "hdfs://master:8020/transaction.snappy.parquet",
    "bronze_path" : "hdfs://master:8020/delta/bronze/transactions",
    "archive_dir" : "hdfs://master:8020/delta/bronze/transactions/archive",
    "database"    : "bronze",
    "table_name"  : "bronze_transactions",
}

def get_spark():
    return (
        SparkSession.builder
        .appName("Bronze_Transactions_Load")
        .master("spark://master:7077")
        .config("spark.jars.packages",
                "io.delta:delta-spark_2.12:3.2.1")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.hadoop.fs.defaultFS", "hdfs://master:8020")
        .config("spark.sql.warehouse.dir", "hdfs://master:8020/user/hive/warehouse")
        .config("spark.hadoop.hive.metastore.uris", "thrift://master:9083")
        .config("spark.databricks.delta.stats.collect", "false")
        .enableHiveSupport()
        .getOrCreate()
    )

def run():
    logger.info("Transactions Bronze pipeline started.")
    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    # Read Parquet
    logger.info(f"Reading from {CONFIG['source_path']}")
    df_raw = spark.read.parquet(CONFIG["source_path"])
    logger.info(f"Extracted {df_raw.count()} rows.")

    # Transform: convert transaction_date to timestamp
    df_bronze = df_raw.withColumn("transaction_date", to_timestamp(col("transaction_date"))) \
                      .withColumn("ingestion_timestamp", current_timestamp())

    
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {CONFIG['database']}")
    spark.sql(f"DROP TABLE IF EXISTS {CONFIG['database']}.{CONFIG['table_name']}")

    # Write as Delta
    logger.info(f"Writing Delta to {CONFIG['bronze_path']}...")
    df_bronze.write.format("delta") \
        .mode("overwrite") \
        .option("delta.columnMapping.mode", "name") \
        .save(CONFIG["bronze_path"])

    # Register in Hive
    spark.sql(f"""
        CREATE TABLE {CONFIG['database']}.{CONFIG['table_name']}
        USING DELTA
        LOCATION '{CONFIG['bronze_path']}'
    """)

    
    count = spark.read.format("delta").load(CONFIG["bronze_path"]).count()
    logger.info(f"SUCCESS: {count} rows in Delta table.")
    spark.read.format("delta").load(CONFIG["bronze_path"]).show(5, truncate=False)

    
    from datetime import datetime
    from subprocess import check_call
    archive_filename = f"transaction_{datetime.now().strftime('%Y%m%d%H%M%S')}.parquet"
    archive_path = f"{CONFIG['archive_dir']}/{archive_filename}"
    check_call(["hdfs", "dfs", "-mkdir", "-p", CONFIG["archive_dir"]], stdout=None, stderr=None)
    check_call(["hdfs", "dfs", "-mv", CONFIG["source_path"], archive_path])
    logger.info(f"Archived source to {archive_path}")

if __name__ == "__main__":
    run()

2026-03-13 15:01:21,984 [INFO] Transactions Bronze pipeline started.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jupyter/.ivy2/cache
The jars for the packages stored in: /home/jupyter/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9eb1de5a-eb60-4463-aac6-b95ca3315363;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 225ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |  

+--------------+-----------+----------+--------+------------+-------------------+--------------+--------------+-----------------------+
|transaction_id|customer_id|product_id|quantity|total_amount|transaction_date   |payment_method|store_type    |ingestion_timestamp    |
+--------------+-----------+----------+--------+------------+-------------------+--------------+--------------+-----------------------+
|TRX000001     |802        |425       |1       |363.4       |2020-07-27 00:00:00|Debit Card    |Physical Store|2026-03-13 15:01:43.642|
|TRX000002     |858        |280       |6       |758.18      |2022-08-10 00:00:00|Credit Card   |Physical Store|2026-03-13 15:01:43.642|
|TRX000003     |658        |694       |9       |748.66      |2020-05-22 00:00:00|Bank Transfer |Online        |2026-03-13 15:01:43.642|
|TRX000004     |516        |930       |4       |933.78      |NULL               |Bank Transfer |Physical Store|2026-03-13 15:01:43.642|
|TRX000005     |368        |104       |10      |

2026-03-13 15:02:22,001 [INFO] Archived source to hdfs://master:8020/delta/bronze/transactions/archive/transaction_20260313150216.parquet
